# 紅酒課程 Part 2：Lasso 特徵選擇與多項式迴歸

這份 Notebook 在 Part 1 前面加入 Lasso 特徵選擇。重點是保持測試集獨立：切分後，所有 `fit()` 都只使用訓練集。

## 1. 匯入工具與讀取紅酒資料

每一列是一瓶紅酒；`quality` 是預測答案，其他 11 欄是原始輸入特徵。

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 相對路徑以 Jupyter 目前工作目錄為起點，不一定是 Notebook 所在目錄。
data_path = Path("../附件/L4 課程範例檔/dataset/winequality-red.csv")
wine = pd.read_csv(data_path)

print(f"wine.shape: {wine.shape}")
wine.head()

wine.shape: (1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## 2. 分離特徵 `X` 與答案 `y`

`X` 儲存 11 個輸入欄位；`y` 儲存每瓶紅酒的正確品質分數。

In [2]:
X = wine.drop(columns="quality")
y = wine["quality"]

print(f"X.shape: {X.shape}")
print(f"y.shape: {y.shape}")

X.shape: (1599, 11)
y.shape: (1599,)


## 3. 先切分訓練集與測試集

Lasso 只能從訓練集決定保留哪些特徵。若先用完整資料選特徵，測試集就會提前參與決策，造成資料洩漏。

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

print(f"訓練特徵：{X_train.shape}")
print(f"測試特徵：{X_test.shape}")

訓練特徵：(1119, 11)
測試特徵：(480, 11)


## 4. 標準化後訓練 Lasso

Lasso 會懲罰係數大小，因此先統一欄位尺度，避免測量單位主導特徵選擇。`alpha=0.1` 越大，通常會把更多係數壓成 0。

In [4]:
selection_scaler = StandardScaler()
X_train_scaled_for_lasso = selection_scaler.fit_transform(X_train)

lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled_for_lasso, y_train)

lasso_coefficients = pd.Series(lasso.coef_, index=X.columns)
print(lasso_coefficients)

fixed acidity           0.000000
volatile acidity       -0.169266
citric acid             0.000000
residual sugar         -0.000000
chlorides              -0.000000
free sulfur dioxide    -0.000000
total sulfur dioxide   -0.000000
density                -0.000000
pH                     -0.000000
sulphates               0.032663
alcohol                 0.253476
dtype: float64


## 5. 將非零係數轉成布林 mask

`lasso.coef_ != 0` 對每個係數進行比較。非零得到 `True`，零得到 `False`。

In [5]:
mask = lasso.coef_ != 0
selected_columns = X.columns[mask]

print(f"mask: {mask}")
print(f"保留特徵數：{mask.sum()} / {len(mask)}")
print(f"保留欄位：{selected_columns.tolist()}")

mask: [False  True False False False False False False False  True  True]
保留特徵數：3 / 11
保留欄位：['volatile acidity', 'sulphates', 'alcohol']


## 6. 訓練集與測試集套用同一個 mask

`loc[:, mask]` 中的 `:` 保留所有 row；`mask` 決定保留哪些 column。兩邊必須使用同一個 mask，才會有相同欄位與順序。

In [6]:
X_train_selected = X_train.loc[:, mask]
X_test_selected = X_test.loc[:, mask]

print(f"X_train_selected.shape: {X_train_selected.shape}")
print(f"X_test_selected.shape: {X_test_selected.shape}")
X_train_selected.head()

X_train_selected.shape: (1119, 3)
X_test_selected.shape: (480, 3)


,volatile acidity,sulphates,alcohol
126,1.330,0.49,10.9
810,0.490,0.47,10.5
635,0.840,0.55,9.7
598,0.585,0.48,9.8
880,0.560,0.49,9.9


## 7. 建立二次多項式特徵

`degree=2` 會加入平方項與兩兩交互項。`include_bias=False` 避免額外常數欄位和線性迴歸的截距重複。

In [7]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_selected)
X_test_poly = poly.transform(X_test_selected)

print(f"原始特徵數：{X_train.shape[1]}")
print(f"Lasso 選擇後特徵數：{X_train_selected.shape[1]}")
print(f"二次多項式特徵數：{X_train_poly.shape[1]}")
print(poly.get_feature_names_out(selected_columns))

原始特徵數：11
Lasso 選擇後特徵數：3
二次多項式特徵數：9
['volatile acidity' 'sulphates' 'alcohol' 'volatile acidity^2'
 'volatile acidity sulphates' 'volatile acidity alcohol' 'sulphates^2'
 'sulphates alcohol' 'alcohol^2']


## 8. 標準化多項式特徵並訓練最後模型

`poly_scaler` 從多項式訓練特徵學習新的平均值與標準差。它和前面給 Lasso 用的 `selection_scaler` 輸入不同，不能混用。

In [8]:
poly_scaler = StandardScaler()
X_train_poly_scaled = poly_scaler.fit_transform(X_train_poly)
X_test_poly_scaled = poly_scaler.transform(X_test_poly)

poly_model = LinearRegression()
poly_model.fit(X_train_poly_scaled, y_train)
y_pred = poly_model.predict(X_test_poly_scaled)

## 9. 用 MSE 與 R² 評估

MSE 越小越好；R² 越接近 1 越好。R² 不是預測正確率。

In [9]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"R²: {r2:.4f}")

prediction_preview = pd.DataFrame({
    "實際品質": y_test.iloc[:10].to_numpy(),
    "預測品質": y_pred[:10],
})
prediction_preview

MSE: 0.4060
R²: 0.3330


,實際品質,預測品質
0,5,5.871771
1,6,5.497926
2,6,6.257258
3,6,5.759786
4,6,6.261545
5,6,6.063932
6,6,6.063177
7,5,5.403896
8,5,5.502805
9,5,5.209993


## 執行後應能回答

- Lasso 為什麼能用來選擇特徵？
- `alpha` 變大時，非零係數的數量通常如何改變？
- 為什麼必須先切分資料，才訓練 Lasso？
- `mask` 儲存什麼，又如何改變 DataFrame 的 column？
- 為什麼本流程有兩個不同的 scaler？
- 特徵變少為什麼不代表測試表現一定變好？